In [1]:
!pip -q install pandas numpy scikit-learn torch transformers accelerate sentence-transformers
!pip -q install huggingface_hub ollama python-dotenv imbalanced-learn

In [2]:
!pip install pandas numpy scikit-learn xgboost lightgbm
!pip install torch transformers datasets accelerate
!pip install sentence-transformers FlagEmbedding
!pip install openai ollama
!pip install optuna 
!pip install xformers # Para hyperparameter tuning

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.7/247.7 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 866.1/866.1 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.7/149.7 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 47.9 MB/s eta 0:00:00
  Created wheel for warc3-wet-clueweb09: filename=warc3_wet_clueweb09-0.2.5-py3-none-any.whl size=18919 sha256=32fa9cad62d32a3d301b825e56e5e5d3414d2e8e0eedb3958d0fdf56f31454fb
  Stored in directory: /root/.cache/pip/wheels/f6/85/c2/9f0f621def52a1d5db7d29984f81e45f9fb6dfeb1a4eb6e31c
  Created wheel for cbor: filename=cbor-1.0.0-cp312-cp312-linux_x86_64.whl size=55021 sha256=ec3e7edc9419b64be0910e635a7ba098cc0e14c3a565e2b429626a66608dcc72
  Stored in directory: /root/.cache/pip/wheels/44/3e/21/a739cbcc331a1ab45c326d6e

In [3]:
import os
import json
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Optional
from dataclasses import dataclass, field

T1	RoBERTa	PlanTL-GOB-ES/roberta-base-bne	125M	768	Español	500 MB	Rápido	████▱	Gob. España, corpus BNE, muy sólido
T1	RoBERTa	PlanTL-GOB-ES/roberta-large-bne	355M	1024	Español	1.4 GB	Medio	█████	Versión large BNE, top para español
T1	RoBERTa	pysentimiento/robertuito-base-uncased	125M	768	Español	500 MB	Rápido	████▱	Tweets español, ideal para texto informal/ironía
T1	RoBERTa	pysentimiento/robertuito-base-cased	125M	768	Español	500 MB	Rápido	████▱	Versión cased de Robertuito
T1	XLM-R	xlm-roberta-large	559M	1024	Multilingüe	2.2 GB	Lento	█████	Versión large, top multilingüe clásico
T1	XLM-R	FacebookAI/xlm-roberta-xl	3.5B	2560	Multilingüe	14 GB	Muy lento	█████	XL, máxima calidad pero requiere mucha VRAM
T1	DeBERTa	microsoft/mdeberta-v3-base	278M	768	Multilingüe	860 MB	Medio	█████	Suele ganar benchmarks de clasificación, muy recomendado
T1	DeBERTa	microsoft/deberta-v3-large	434M	1024	Inglés	1.5 GB	Lento	█████	Top en GLUE/SuperGLUE, solo inglés
T1	XLM-R	joeddav/xlm-roberta-large-xnli	559M	1024	Multilingüe	2.2 GB	Lento	█████	Zero-shot NLI classification, excelente
T1	ModernBERT	answerdotai/ModernBERT-large	395M	1024	Inglés	1.3 GB	Medio	█████	2024, supera DeBERTa en muchos benchmarks
T2	BERT	dccuchile/bert-base-spanish-wwm-cased	110M	768	Español	440 MB	Rápido	████▱	BETO — clásico español, whole word masking
T2	BERT	dccuchile/bert-base-spanish-wwm-uncased	110M	768	Español	440 MB	Rápido	███▱▱	BETO uncased, útil para texto informal
T2	RoBERTa	bertin-project/bertin-roberta-base-spanish	125M	768	Español	500 MB	Rápido	███▱▱	Datos españoles variados, mC4
T2	RoBERTa	BSC-TeMU/roberta-base-bne-capitel	125M	768	Español	500 MB	Rápido	████▱	Fine-tuned en CAPITEL (NER+POS español)
T2	XLM-R	xlm-roberta-base	278M	768	Multilingüe	1.1 GB	Medio	████▱	Meta, 100 idiomas, robusto multilingüe
T2	DeBERTa	microsoft/deberta-v3-base	183M	768	Inglés	710 MB	Medio	████▱	Solo inglés pero excelente arquitectura
T2	ALBERT	albert-xxlarge-v2	223M	4096	Inglés	890 MB	Lento	████▱	Grande pero eficiente en parámetros
T2	ELECTRA	mrm8488/electricidad-base-discriminator	110M	768	Español	440 MB	Rápido	███▱▱	ELECTRA para español, eficiente en datos
T2	GPT-2	PlanTL-GOB-ES/gpt2-large-bne	774M	1280	Español	1.5 GB	Medio	███▱▱	GPT-2 large español, generativo
T2	Longformer	allenai/longformer-base-4096	148M	768	Inglés	570 MB	Medio	███▱▱	4096 tokens contexto, textos largos
T2	Longformer	markussagen/xlm-roberta-longformer-base-4096	278M	768	Multilingüe	1.1 GB	Medio	███▱▱	XLM-R + Longformer, multilingüe largo
T2	XLM-R	symanto/sn-xlm-roberta-base-snli-mnli-anli-xnli	278M	768	Multilingüe	1.1 GB	Medio	████▱	Fine-tuned en NLI, bueno para clasificación zero-shot
T2	ModernBERT	answerdotai/ModernBERT-base	149M	768	Inglés	560 MB	Rápido	████▱	2024, Flash Attention, 8192 tokens, alternating attn
T3	BERT	bert-base-multilingual-cased	110M	768	Multilingüe	680 MB	Rápido	███▱▱	mBERT original, 104 idiomas, baseline clásico
T3	BERT	bert-base-multilingual-uncased	110M	768	Multilingüe	680 MB	Rápido	██▱▱▱	Versión uncased de mBERT, peor para español
T3	ALBERT	albert-base-v2	12M	768	Inglés	47 MB	Muy rápido	██▱▱▱	Ultra ligero, parameter sharing
T3	ELECTRA	google/electra-base-discriminator	110M	768	Inglés	440 MB	Rápido	███▱▱	Replaced token detection, eficiente
T3	DistilBERT	distilbert-base-multilingual-cased	66M	768	Multilingüe	540 MB	Muy rápido	███▱▱	40% más rápido que mBERT, 97% rendimiento
T3	GPT-2	PlanTL-GOB-ES/gpt2-base-bne	117M	768	Español	500 MB	Rápido	███▱▱	GPT-2 español, autoregressive, BNE
LLMs para Predicción de Niveles CEFR en Español

24 modelos · Clasificación A1/A2/B1/B2/C1 · NivELE 2026

▸ Ver prompt CEFR de ejemplo para zero-shot

Buscar modelo...
Todos
Tier 1
Tier 2
Tier 3
Todos
HF
HF / Ollama
API
T1	meta-llama/Llama-3.1-70B-Instruct	70B	Multilingüe	HF / Ollama	Zero/Few-shot	40+ GB	Muy lento	█████	Excelente comprensión del español, top en razonamiento lingüístico
T1	meta-llama/Llama-3.1-8B-Instruct	8B	Multilingüe	HF / Ollama	Zero/Few-shot + LoRA	6 GB	Medio	████▱	Buen español, cabe en T4, fine-tuneable con LoRA
T1	Qwen/Qwen2.5-72B-Instruct	72B	Multilingüe	HF / Ollama	Zero/Few-shot	40+ GB	Muy lento	█████	Alibaba, excelente multilingüe, competitivo con GPT-4
T1	Qwen/Qwen2.5-7B-Instruct	7B	Multilingüe	HF / Ollama	Zero/Few-shot + LoRA	5 GB	Medio	████▱	Sorprendente calidad en español para su tamaño
T1	google/gemma-2-9b-it	9B	Multilingüe	HF / Ollama	Zero/Few-shot + LoRA	6 GB	Medio	████▱	Google, fuerte en tareas de clasificación lingüística
T1	google/gemma-2-27b-it	27B	Multilingüe	HF / Ollama	Zero/Few-shot	18 GB	Lento	█████	Cabe en A100 40GB, muy bueno en español
T1	mistralai/Mixtral-8x7B-Instruct-v0.1	46.7B MoE	Multilingüe	HF / Ollama	Zero/Few-shot	26 GB	Lento	█████	Mixture of Experts, calidad/eficiencia excelente
T1	OpenAI GPT-4o	?	Multilingüe	API	Zero/Few-shot	API	Rápido	█████	Top en comprensión lingüística, ideal para CEFR con prompt engineering
T1	OpenAI GPT-4o-mini	?	Multilingüe	API	Zero/Few-shot	API	Muy rápido	████▱	10x más barato que GPT-4o, muy bueno en clasificación
T1	Anthropic Claude Sonnet	?	Multilingüe	API	Zero/Few-shot	API	Rápido	█████	Excelente español, gran capacidad de análisis lingüístico
T1	Google Gemini 1.5 Pro	?	Multilingüe	API	Zero/Few-shot	API	Rápido	█████	1M tokens contexto, ideal para muchos few-shot examples
T1	microsoft/mdeberta-v3-baseENCODER	278M	Multilingüe	HF	Fine-tune completo	2 GB	Muy rápido	█████	NO es LLM pero suele SUPERAR LLMs en clasificación supervisada
T1	PlanTL-GOB-ES/roberta-large-bneENCODER	355M	Español	HF	Fine-tune completo	2.5 GB	Rápido	█████	NO es LLM pero top para clasificación en español con datos
T2	mistralai/Mistral-7B-Instruct-v0.3	7B	Multilingüe	HF / Ollama	Zero/Few-shot + LoRA	5 GB	Medio	████▱	Rápido y eficiente, buen español
T2	clibrain/Llama-2-7b-ft-instruct-es	7B	Español	HF	Zero/Few-shot + LoRA	5 GB	Medio	███▱▱	Llama-2 fine-tuned en instrucciones en español
T2	projecte-aina/aguila-7b	7B	Español + Catalán	HF	Zero/Few-shot + LoRA	5 GB	Medio	███▱▱	BSC, entrenado en español/catalán, corpus peninsular
T2	HiTZ/latxa-7b-v1.2	7B	Español + Euskera	HF	Few-shot + LoRA	5 GB	Medio	███▱▱	Basado en Llama, datos españoles y vascos
T2	meta-llama/Llama-3.2-3B-Instruct	3B	Multilingüe	HF / Ollama	Zero/Few-shot + LoRA	2.5 GB	Rápido	███▱▱	Ligero, viable en CPU, español aceptable
T2	Qwen/Qwen2.5-3B-Instruct	3B	Multilingüe	HF / Ollama	Zero/Few-shot + LoRA	2.5 GB	Rápido	███▱▱	Alibaba, buen español para 3B
T2	Mistral Large	?	Multilingüe	API	Zero/Few-shot	API	Rápido	████▱	Empresa francesa, buen soporte lenguas romances
T3	PlanTL-GOB-ES/gpt2-large-bne	774M	Español	HF	Fine-tune completo	2 GB	Rápido	██▱▱▱	GPT-2 español BNE, ligero, fine-tuneable fácilmente
T3	Qwen/Qwen2.5-1.5B-Instruct	1.5B	Multilingüe	HF / Ollama	Few-shot + LoRA	1.5 GB	Muy rápido	██▱▱▱	Ultra ligero, mínimo viable para clasificación
T3	microsoft/phi-3-mini-4k-instruct	3.8B	Multilingüe	HF / Ollama	Zero/Few-shot + LoRA	3 GB	Rápido	███▱▱	Microsoft, eficiente pero español más débil
T3	google/gemma-2-2b-it	2B	Multilingüe	HF / Ollama	Few-shot + LoRA	2 GB	Muy rápido	██▱▱▱	Google, mínimo viable, rápido de iterar

Modelos de Embedding en Ollama
14 modelos · Para clasificación CEFR en español · ollama embed

▸ Ver código Python para usar estos embeddings

Buscar modelo...
Todos
Tier 1
Tier 2
Tier 3
Todos
Embedding
LLM → Embed
T1	Embedding	mxbai-embed-large	1024	670 MB	Rápido	█████	ollama pull mxbai-embed-large	Mejor embedding en Ollama, mixedbread-ai, top MTEB
T1	Embedding	bge-m3	1024	1.2 GB	Medio	█████	ollama pull bge-m3	BAAI, excelente español, sparse+dense, recomendado para CEFR
T1	Embedding	snowflake-arctic-embed2	1024	1.2 GB	Medio	████▱	ollama pull snowflake-arctic-embed2	Snowflake, matryoshka (reduce dim sin perder calidad)
T2	Embedding	nomic-embed-text	768	274 MB	Muy rápido	████▱	ollama pull nomic-embed-text	Ultra ligero, ideal para iterar rápido, buen multilingüe
T2	Embedding	snowflake-arctic-embed	1024	670 MB	Rápido	████▱	ollama pull snowflake-arctic-embed	Versión original Snowflake, sólida
T2	Embedding	bge-large	1024	670 MB	Rápido	████▱	ollama pull bge-large	BAAI, inglés-centric pero funcional en español
T2	LLM → Embed	llama3.1:8b	4096	4.7 GB	Lento	████▱	ollama pull llama3.1:8b	Puedes extraer embeddings del último hidden state, buen español
T2	LLM → Embed	gemma2:9b	3584	5.4 GB	Lento	████▱	ollama pull gemma2:9b	Google, embeddings ricos semánticamente
T2	LLM → Embed	qwen2.5:7b	3584	4.7 GB	Lento	████▱	ollama pull qwen2.5:7b	Alibaba, muy bueno en español, embeddings densos
T3	Embedding	all-minilm	384	46 MB	Muy rápido	███▱▱	ollama pull all-minilm	Mínimo viable, 46MB, bueno para prototipar
T3	LLM → Embed	mistral:7b	4096	4.1 GB	Lento	███▱▱	ollama pull mistral:7b	Embeddings decentes, más útil como clasificador directo
T3	LLM → Embed	llama3.2:3b	3072	2.0 GB	Medio	███▱▱	ollama pull llama3.2:3b	Ligero, embeddings aceptables para clasificación
T3	LLM → Embed	qwen2.5:3b	2048	1.9 GB	Medio	███▱▱	ollama pull qwen2.5:3b	Ligero, buen español para su tamaño
T3	LLM → Embed	qwen2.5:1.5b	1536	986 MB	Rápido	██▱▱▱	ollama pull qwen2.5:1.5b	Ultra ligero, mínimo viable como LLM embed

In [4]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))
        
@dataclass
class Config:
    """Configuración centralizada del pipeline."""
    # Rutas
    train_path: str = "/kaggle/input/datasets/ymlopez/training-corpus/training_corpus_caes.csv" #"/kaggle/input/datasets/ymlopez/train-caes/train_caes.csv"
    test_path: str = "/kaggle/input/competitions/nivele-2026/test.csv"
    output_dir: str = "output"
    models_dir: str = "models"

    # Labels CEFR
    labels: list = field(default_factory=lambda: ["A1", "A2", "B1", "B2", "C1", "C2"])
    label2id: dict = field(default_factory=lambda: {
        "A1": 0, "A2": 1, "B1": 2, "B2": 3, "C1": 4, "C2": 5
    })
    id2label: dict = field(default_factory=lambda: {
        0: "A1", 1: "A2", 2: "B1", 3: "B2", 4: "C1", 5: "C2"
    })

    # Embeddings
    embedding_model: str = "allenai/longformer-base-4096" #"nomic-ai/nomic-embed-text-v1" #"microsoft/mdeberta-v3-base" #"sentence-transformers/LaBSE" #"BAAI/bge-m3" #"google/embeddinggemma-300M" #"nomic-ai/nomic-embed-text-v1.5" #"intfloat/multilingual-e5-large" #"nomic-ai/nomic-embed-text-v1"
    embedding_batch_size: int = 32

    # Fine-tuning
    encoder_model: str = "sentence-transformers/paraphrase-multilingual-768-v2"
    max_length: int = 512
    ft_epochs: int = 10
    ft_batch_size: int = 8
    ft_lr: float = 2e-5
    ft_warmup_ratio: float = 0.1
    ft_weight_decay: float = 0.01

    # LLM
    openai_model: str = "gpt-5.2"
    ollama_model: str = "llama3:8b"

    # General
    seed: int = 42
    n_folds: int = 5
    device: str = "auto"  # auto, cuda, cpu
        
        
cfg = Config()
os.makedirs(cfg.output_dir, exist_ok=True)
os.makedirs(cfg.models_dir, exist_ok=True)

/kaggle/input/competitions/nivele-2026/random_submission.csv
/kaggle/input/competitions/nivele-2026/test.csv
/kaggle/input/datasets/ymlopez/training-corpus/training_corpus_caes.csv


In [ ]:
from huggingface_hub import login
login(token="")

In [6]:
# ============================================================
# 1. CARGA Y PREPROCESAMIENTO DE DATOS
# ============================================================

def load_data(path: str) -> pd.DataFrame:
    """Carga datos en formato TSV (label\\ttext)."""
    df = pd.read_csv(path, names=["label", "text"], header=0)
    df["text"] = df["text"].astype(str).str.strip()
    df["label"] = df["label"].str.strip().str.upper()
    print(f"Cargados {len(df)} ejemplos desde {path}")
    print(f"Distribución:\n{df['label'].value_counts().sort_index()}\n")
    return df

def load_data_pred(path: str) -> pd.DataFrame:
    """Carga datos en formato TSV (label\\ttext)."""
    df = pd.read_csv(path, names=["text"], header=0)
    df["text"] = df["text"].astype(str).str.strip()
    #df["label"] = df["label"].str.strip().str.upper()
    print(f"Cargados {len(df)} ejemplos desde {path}")
    #print(f"Distribución:\n{df['label'].value_counts().sort_index()}\n")
    return df


def extract_linguistic_features(text: str) -> dict:
    """
    Extrae features lingüísticas relevantes para la complejidad CEFR.
    Estas features capturan indicadores gramaticales, léxicos y discursivos.
    """
    import re

    words = text.split()
    sentences = re.split(r'[.!?]+', text)
    sentences = [s.strip() for s in sentences if s.strip()]

    n_words = len(words)
    n_sentences = max(len(sentences), 1)
    n_chars = len(text)

    # Léxico
    unique_words = set(w.lower() for w in words)
    ttr = len(unique_words) / max(n_words, 1)  # Type-Token Ratio

    # Longitud
    avg_word_len = np.mean([len(w) for w in words]) if words else 0
    avg_sent_len = n_words / n_sentences
    max_sent_len = max((len(s.split()) for s in sentences), default=0)

    # Complejidad gramatical (indicadores)
    subjunctive_markers = len(re.findall(
        r'\b(que|aunque|para que|sin que|antes de que|ojalá|quizás|tal vez)\b',
        text.lower()
    ))
    conditional_markers = len(re.findall(
        r'\b(si|sería|podría|debería|habría|tendría)\b', text.lower()
    ))
    connectors = len(re.findall(
        r'\b(sin embargo|no obstante|además|por lo tanto|en consecuencia|'
        r'por otro lado|en primer lugar|en conclusión|asimismo|de hecho|'
        r'a pesar de|en cambio|por consiguiente|cabe destacar)\b',
        text.lower()
    ))
    relative_clauses = len(re.findall(
        r'\b(que|quien|quienes|cual|cuales|cuyo|cuya|donde)\b', text.lower()
    ))

    # Tiempos verbales (aproximación)
    past_markers = len(re.findall(
        r'\b\w+(ó|aron|ieron|aba|aban|ía|ían)\b', text.lower()
    ))
    future_markers = len(re.findall(
        r'\b\w+(ré|rás|rá|remos|réis|rán)\b', text.lower()
    ))

    # Puntuación y formato
    n_commas = text.count(',')
    n_semicolons = text.count(';')
    n_paragraphs = text.count('\n\n') + 1
    has_greeting = int(bool(re.search(r'(estimad|hola|querido)', text.lower())))
    has_farewell = int(bool(re.search(
        r'(atentamente|saludos|adiós|un abrazo|cordialmente)', text.lower()
    )))

    return {
        "n_words": n_words,
        "n_sentences": n_sentences,
        "n_chars": n_chars,
        "n_paragraphs": n_paragraphs,
        "ttr": round(ttr, 4),
        "avg_word_len": round(avg_word_len, 2),
        "avg_sent_len": round(avg_sent_len, 2),
        "max_sent_len": max_sent_len,
        "subjunctive_markers": subjunctive_markers,
        "conditional_markers": conditional_markers,
        "connectors": connectors,
        "relative_clauses": relative_clauses,
        "past_markers": past_markers,
        "future_markers": future_markers,
        "n_commas": n_commas,
        "n_semicolons": n_semicolons,
        "has_greeting": has_greeting,
        "has_farewell": has_farewell,
        "punct_density": round((n_commas + n_semicolons) / max(n_sentences, 1), 3),
        "connector_density": round(connectors / max(n_sentences, 1), 3),
    }


def build_feature_matrix(df: pd.DataFrame) -> pd.DataFrame:
    """Construye la matriz de features lingüísticas para todo el dataset."""
    features = df["text"].apply(extract_linguistic_features)
    feat_df = pd.DataFrame(features.tolist())
    print(f"Features lingüísticas extraídas: {feat_df.shape[1]} columnas")
    return feat_df

In [7]:
# ============================================================
# 2. ENFOQUE 1: EMBEDDINGS + CLASIFICADORES
# ============================================================

def get_embeddings(texts: list[str], model_name: str = None) -> np.ndarray:
    """
    Genera embeddings densos con sentence-transformers o FlagEmbedding.
    Soporta: bge-m3, multilingual-e5-large, etc.
    """
    model_name = model_name or cfg.embedding_model

    if "bge-m3" in model_name:
        from FlagEmbedding import BGEM3FlagModel
        model = BGEM3FlagModel(model_name, use_fp16=True)
        embeddings = model.encode(
            texts,
            batch_size=cfg.embedding_batch_size,
            max_length=cfg.max_length
        )["dense_vecs"]
    else:
        from sentence_transformers import SentenceTransformer
        model = SentenceTransformer(model_name, trust_remote_code=True)
        embeddings = model.encode(
            texts,
            batch_size=cfg.embedding_batch_size,
            show_progress_bar=True,
            normalize_embeddings=True
        )

    return np.array(embeddings)


def train_classifiers_and_predict(X_train, y_train, X_val, y_val, X_test):
    """
    Entrena múltiples clasificadores MULTICLASE (5 niveles CEFR)
    y devuelve resultados ordenados por Macro F1.
    """
    from sklearn.linear_model import LogisticRegression
    from sklearn.svm import SVC
    from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
    from xgboost import XGBClassifier
    from sklearn.metrics import classification_report, f1_score, confusion_matrix
    from sklearn.preprocessing import StandardScaler

    n_classes = len(cfg.labels)

    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_val_s = scaler.transform(X_val)
    X_test_s = scaler.transform(X_test)
    
    # Calcular pesos de clase para XGBoost (no acepta class_weight="balanced")
    from sklearn.utils.class_weight import compute_sample_weight
    xgb_sample_weights = compute_sample_weight("balanced", y_train)

    classifiers = {
        "LogReg": LogisticRegression(
            max_iter=2000, C=1.0, class_weight="balanced",
            multi_class="multinomial", solver="lbfgs",
            random_state=cfg.seed
        ),
        "SVM_RBF": SVC(
            kernel="rbf", C=10, gamma="scale",
            class_weight="balanced", random_state=cfg.seed,
            probability=True, decision_function_shape="ovr"
        ),
        "SVM_Linear": SVC(
            kernel="linear", C=1.0,
            class_weight="balanced", random_state=cfg.seed,
            probability=True, decision_function_shape="ovr"
        ),
        "XGBoost": XGBClassifier(
            n_estimators=300, max_depth=6, learning_rate=0.1,
            objective="multi:softprob", num_class=n_classes,
            eval_metric="mlogloss",
            random_state=cfg.seed
        ),
        "RandomForest": RandomForestClassifier(
            n_estimators=300, max_depth=None,
            class_weight="balanced", random_state=cfg.seed
        ),
        "GradientBoosting": GradientBoostingClassifier(
            n_estimators=200, max_depth=5, learning_rate=0.1,
            random_state=cfg.seed
        ),
    }

    results = {}
    for name, clf in classifiers.items():
        print(f"\n{'='*50}")
        print(f"Entrenando: {name} ({n_classes} clases)")
        print(f"{'='*50}")

        # XGBoost necesita sample_weight en vez de class_weight
        if name == "XGBoost":
            clf.fit(X_train_s, y_train, sample_weight=xgb_sample_weights)
        else:
            clf.fit(X_train_s, y_train)

        y_pred = clf.predict(X_val_s)

        # Métricas multiclase
        f1_macro = f1_score(y_val, y_pred, average="macro")
        f1_weighted = f1_score(y_val, y_pred, average="weighted")
        cm = confusion_matrix(y_val, y_pred, labels=list(range(n_classes)))
        report = classification_report(
            y_val, y_pred,
            target_names=cfg.labels,
            labels=list(range(n_classes)),
            digits=4,
            zero_division=0
        )
        print(report)
        print(f"Confusion matrix:\n{cm}\n")

        # Probabilidades por clase (n_samples x n_classes)
        probs = None
        if hasattr(clf, "predict_proba"):
            probs = clf.predict_proba(X_val_s)
            assert probs.shape[1] == n_classes, (
                f"Esperadas {n_classes} columnas, got {probs.shape[1]}"
            )

        
        y_pred_test = clf.predict(X_test_s)  
        
        probs_test = None
        if hasattr(clf, "predict_proba"):
            probs_test = clf.predict_proba(X_test_s)
            assert probs_test.shape[1] == n_classes, (
                f"Esperadas {n_classes} columnas, got {probs_test.shape[1]}"
            )
        
        results[name] = {
            "model": clf,
            "scaler": scaler,
            "f1_macro": f1_macro,
            "f1_weighted": f1_weighted,
            "predictions": y_pred,
            "predictions_test" :  y_pred_test,
            "probabilities": probs,
            "probabilities_test": probs_test,
            "confusion_matrix": cm,
        }

    # Ordenar por F1 Macro (métrica principal de NivELE)
    results = dict(sorted(
        results.items(), key=lambda x: x[1]["f1_macro"], reverse=True
    ))
    print("\n📊 Ranking de clasificadores (Macro F1):")
    for i, (name, r) in enumerate(results.items(), 1):
        print(f"  {i}. {name}: MacroF1={r['f1_macro']:.4f} | "
              f"WeightedF1={r['f1_weighted']:.4f}")

    return results


In [8]:
def run_embedding_pipeline(df_train, df_val, df_test):
    """Pipeline completo: embeddings + features lingüísticas + clasificadores multiclase."""
    from sklearn.preprocessing import LabelEncoder
    from sklearn.model_selection import StratifiedKFold, cross_val_score

    print("\n" + "="*60)
    print("ENFOQUE 1: EMBEDDINGS + CLASIFICADORES (5 clases CEFR)")
    print("="*60)

    # Codificar labels con orden ordinal CEFR
    le = LabelEncoder()
    le.classes_ = np.array(cfg.labels)  # A1=0, A2=1, B1=2, B2=3, C1=4, C2=5
    y_train = le.transform(df_train["label"])
    y_val = le.transform(df_val["label"])

    # Verificar distribución de clases
    print(f"\nDistribución train: {dict(zip(*np.unique(y_train, return_counts=True)))}")
    print(f"Distribución val:   {dict(zip(*np.unique(y_val, return_counts=True)))}")

    n_classes_train = len(np.unique(y_train))
    n_classes_val = len(np.unique(y_val))
    if n_classes_train < len(cfg.labels):
        print(f"⚠️  Solo {n_classes_train} clases en train (faltan clases)")
    if n_classes_val < len(cfg.labels):
        print(f"⚠️  Solo {n_classes_val} clases en val (faltan clases)")

    # 1a. Embeddings densos
    print("\n🔄 Generando embeddings...")
    emb_train = get_embeddings(df_train["text"].tolist())
    emb_val = get_embeddings(df_val["text"].tolist())
    emb_test = get_embeddings(df_test["text"].tolist())
    
    # 1b. Features lingüísticas
    print("🔄 Extrayendo features lingüísticas...")
    feat_train = build_feature_matrix(df_train).values
    feat_val = build_feature_matrix(df_val).values
    feat_test = build_feature_matrix(df_test).values

    # 1c. Concatenar embeddings + features
    X_train = np.hstack([emb_train, feat_train])
    X_val = np.hstack([emb_val, feat_val])
    X_test = np.hstack([emb_test, feat_test])
    
    print(f"\nDimensión final: {X_train.shape[1]} "
          f"(embeddings={emb_train.shape[1]} + features={feat_train.shape[1]})")

    # 1d. Cross-validation estratificado (respeta las 5 clases)
    print(f"\n🔄 Cross-validation estratificado ({cfg.n_folds} folds)...")
    skf = StratifiedKFold(
        n_splits=cfg.n_folds, shuffle=True, random_state=cfg.seed
    )
    # Verificar que cada fold tiene todas las clases
    for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
        fold_classes = np.unique(y_train[val_idx])
        if len(fold_classes) < len(cfg.labels):
            print(f"  ⚠️  Fold {fold_idx}: solo {len(fold_classes)} clases en val")

    # 1e. Entrenar clasificadores
    results = train_classifiers_and_predict(X_train, y_train, X_val, y_val, X_test)

    return results, le


In [9]:
"""Ejecuta el pipeline completo."""
from sklearn.model_selection import train_test_split

print("🚀 NivELE Pipeline - Clasificación CEFR de textos ELE")
print("="*60)
# Cargar datos
#df_train = load_data(cfg.train_path)
#df_val = load_data_pred(cfg.test_path)
df = load_data(cfg.train_path)
# Split estratificado
df_train, df_val = train_test_split(
    df, test_size=0.4, stratify=df["label"], random_state=cfg.seed
)
df_test = load_data_pred(cfg.test_path)

🚀 NivELE Pipeline - Clasificación CEFR de textos ELE
Cargados 91 ejemplos desde /kaggle/input/datasets/ymlopez/training-corpus/training_corpus_caes.csv
Distribución:
label
A1    20
A2    19
B1    17
B2    18
C1    11
C2     6
Name: count, dtype: int64

Cargados 316 ejemplos desde /kaggle/input/competitions/nivele-2026/test.csv


In [10]:
# ── Enfoque 1: Embeddings + Clasificadores ──
clf_results, le = run_embedding_pipeline(df_train, df_val, df_test)
best_clf_name = list(clf_results.keys())[0]


ENFOQUE 1: EMBEDDINGS + CLASIFICADORES (5 clases CEFR)

Distribución train: {np.int64(0): np.int64(12), np.int64(1): np.int64(11), np.int64(2): np.int64(10), np.int64(3): np.int64(11), np.int64(4): np.int64(6), np.int64(5): np.int64(4)}
Distribución val:   {np.int64(0): np.int64(8), np.int64(1): np.int64(8), np.int64(2): np.int64(7), np.int64(3): np.int64(7), np.int64(4): np.int64(5), np.int64(5): np.int64(2)}

🔄 Generando embeddings...


No sentence-transformers model found with name allenai/longformer-base-4096. Creating a new one with mean pooling.


config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/597M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/271 [00:00<?, ?it/s]

...

🔄 Extrayendo features lingüísticas...
Features lingüísticas extraídas: 20 columnas
Features lingüísticas extraídas: 20 columnas
Features lingüísticas extraídas: 20 columnas

Dimensión final: 788 (embeddings=768 + features=20)

🔄 Cross-validation estratificado (5 folds)...
  ⚠️  Fold 4: solo 5 clases en val


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 4 members, which is less than n_splits=5.
  warnings.warn(



Entrenando: LogReg (6 clases)
              precision    recall  f1-score   support

          A1     0.7143    0.6250    0.6667         8
          A2     0.5556    0.6250    0.5882         8
          B1     0.2500    0.4286    0.3158         7
          B2     0.7500    0.4286    0.5455         7
          C1     0.3333    0.2000    0.2500         5
          C2     1.0000    1.0000    1.0000         2

    accuracy                         0.5135        37
   macro avg     0.6005    0.5512    0.5610        37
weighted avg     0.5628    0.5135    0.5221        37

Confusion matrix:
[[5 1 1 1 0 0]
 [2 5 1 0 0 0]
 [0 2 3 0 2 0]
 [0 0 4 3 0 0]
 [0 1 3 0 1 0]
 [0 0 0 0 0 2]]


Entrenando: SVM_RBF (6 clases)


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


              precision    recall  f1-score   support

          A1     0.6667    0.5000    0.5714         8
          A2     0.5000    0.6250    0.5556         8
          B1     0.2222    0.2857    0.2500         7
          B2     0.4286    0.4286    0.4286         7
          C1     0.6667    0.4000    0.5000         5
          C2     1.0000    1.0000    1.0000         2

    accuracy                         0.4865        37
   macro avg     0.5807    0.5399    0.5509        37
weighted avg     0.5195    0.4865    0.4937        37

Confusion matrix:
[[4 2 1 1 0 0]
 [2 5 1 0 0 0]
 [0 2 2 3 0 0]
 [0 1 2 3 1 0]
 [0 0 3 0 2 0]
 [0 0 0 0 0 2]]


Entrenando: SVM_Linear (6 clases)
              precision    recall  f1-score   support

          A1     0.7143    0.6250    0.6667         8
          A2     0.5000    0.6250    0.5556         8
          B1     0.1818    0.2857    0.2222         7
          B2     0.6667    0.2857    0.4000         7
          C1     0.2500    0.2000    0.22

In [11]:
# Aquí usar el mejor modelo/ensemble para predecir
predictions = clf_results['RandomForest']['predictions_test']
labels = [cfg.id2label[p] for p in predictions]
output_path = "submission.csv"
submission = pd.DataFrame({
    "id": range(len(labels)),
    "label": labels
})
submission.to_csv(output_path, index=False)
print(f"\n📄 Submission guardada: {output_path}")
print(f"   Total predicciones: {len(labels)}")
print(f"   Distribución:\n{submission['label'].value_counts().sort_index()}")


📄 Submission guardada: submission.csv
   Total predicciones: 316
   Distribución:
label
A1    132
A2     19
B1     55
B2    105
C2      5
Name: count, dtype: int64
